<img src="https://raw.githubusercontent.com/OmarLi-360/powerbi-tableau-analista/main/assets/logo-drbits.jpg" alt="Dr. Bits" width="170">

# Clave 8 · Publicar, gobernar y portafolio
**Analista de datos · Power BI + Tableau** — Seguridad por filas y qué publicar dónde · ⏱ 20 min

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OmarLi-360/powerbi-tableau-analista/blob/main/notebooks/08_publicar_y_gobernar.ipynb) &nbsp; [Volver al tablero](https://omarli-360.github.io/powerbi-tableau-analista/#clave-08)

<sub>Si al abrir un enlace Google muestra «Redirect Notice», haz clic en el enlace azul de esa página: es un aviso de seguridad de Google, no un error.</sub>

**Al terminar podrás:**
1. Comprobar que cada rol de seguridad ve solo sus filas.
2. Verificar que la suma de los roles da el total.
3. Decidir dónde publicar cada tablero.

---
### Cómo usar este cuaderno (no necesitas programar)
1. **Guarda tu copia:** *Archivo › Guardar una copia en Drive*. Así tus cambios no se pierden. Si no lo haces, no pasa nada grave: el original está protegido y nadie puede dañarlo desde aquí, pero lo que cambies se borra al cerrar.
2. **Ejecuta todo:** *Entorno de ejecución › Ejecutar todas* (Ctrl+F9). Si Colab avisa que el cuaderno no es de Google, elige *Ejecutar de todos modos*.
3. **Juega con los controles:** las celdas con listas desplegables a la derecha se pueden cambiar. Elige una opción y pulsa el botón ▶ de esa celda para redibujar.

El código está oculto a propósito: aquí importa entender el dato y el gráfico. Si tienes curiosidad, haz doble clic en una celda para verlo.

In [ ]:
# @title Preparar el entorno (se ejecuta sola con «Ejecutar todas»)
import os, sys, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, Markdown, HTML
warnings.filterwarnings("ignore")

EN_COLAB = "google.colab" in sys.modules
REPO = "OmarLi-360/powerbi-tableau-analista"
BASE_URL = f"https://raw.githubusercontent.com/{REPO}/main/data/"
BASE_LOCAL = "../data/"

def ruta(nombre):
    local = os.path.join(BASE_LOCAL, nombre)
    return local if os.path.exists(local) else BASE_URL + nombre

# --- formatos colombianos -------------------------------------------------
def miles(x):    return f"{x:,.0f}".replace(",", ".")
def pesos(x):    return "$ " + miles(x)
def millones(x): return "$ " + f"{x/1e6:,.0f}".replace(",", ".") + " M"
def porc(v, signo=False):
    t = f"{v:+.1%}" if signo else f"{v:.1%}"
    return t.replace(".", ",").replace("%", " %")
MESES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
MM = "Millones de pesos"

# --- plantilla de gráficos Dr. Bits ----------------------------------------
AZUL, TEAL, MAGENTA, GRIS = "#4A5AD8", "#0E9BB8", "#D0336F", "#9AA3BF"
BIEN, MAL = "#15784A", "#B42318"
ejes = dict(gridcolor="#E4E8F2", zeroline=False, linecolor="#C9D0E2", exponentformat="none", separatethousands=True)
pio.templates["drbits"] = go.layout.Template(layout=dict(
    font=dict(family="Source Sans 3, Segoe UI, Arial, sans-serif", size=14, color="#11153A"),
    title=dict(font=dict(size=18, color="#00006A"), x=0, xanchor="left"),
    colorway=[AZUL, TEAL, MAGENTA], plot_bgcolor="white", paper_bgcolor="white",
    xaxis=ejes, yaxis=ejes, separators=",.", hoverlabel=dict(font_size=13),
    margin=dict(t=80, l=10, r=30, b=40)))
pio.templates.default = "plotly_white+drbits"

def eje_meses(fig, fechas, cada=None):
    """Etiquetas de meses en español en ejes de tiempo (Plotly las pone en inglés)."""
    f = pd.to_datetime(pd.Series(sorted(set(pd.to_datetime(list(fechas))))))
    paso = cada or (1 if len(f) <= 13 else 3)
    f = f.iloc[::paso]
    fig.update_xaxes(tickvals=list(f), ticktext=[f"{MESES[d.month - 1]} {d.year}" for d in f])

def dar_aire(fig, maximo, eje="x"):
    (fig.update_xaxes if eje == "x" else fig.update_yaxes)(range=[0, maximo * 1.22])

# --- revisor de retos -----------------------------------------------------
def revisar(respuesta, esperado, explicacion=""):
    if respuesta == "Elige una opción":
        display(Markdown("**Elige una opción en la lista de la derecha** y vuelve a ejecutar esta celda con ▶."))
    elif str(respuesta) == str(esperado):
        display(Markdown(f"✅ **Correcto.** {explicacion}"))
    else:
        display(Markdown(f"❌ **Todavía no.** Revisa el gráfico de arriba e inténtalo otra vez. *(Elegiste: {respuesta})*"))

# --- datos de Distribuidora Andina ----------------------------------------
ventas = pd.read_csv(ruta("ventas.csv"), parse_dates=["fecha"])
clientes = pd.read_csv(ruta("clientes.csv"))
productos = pd.read_csv(ruta("productos.csv"))
vendedores = pd.read_csv(ruta("vendedores.csv"))
sedes = pd.read_csv(ruta("sedes.csv"))
metas = pd.read_csv(ruta("metas.csv"))
df = (ventas
      .merge(sedes[["id_sede", "ciudad"]], on="id_sede", validate="many_to_one")
      .merge(productos[["sku", "producto", "categoria"]], on="sku", validate="many_to_one")
      .merge(clientes[["id_cliente", "razon_social", "segmento"]], on="id_cliente", validate="many_to_one")
      .merge(vendedores[["id_vendedor", "nombre_vendedor"]], on="id_vendedor", validate="many_to_one")
      .assign(anio=lambda d: d.fecha.dt.year, mes=lambda d: d.fecha.dt.month))
validas = df[df["estado"] != "Anulada"]          # las anuladas no son venta
ORDEN_CIUDADES = (validas[validas.anio == 2025].groupby("ciudad")["subtotal"].sum()
                  .sort_values(ascending=False).index.tolist())
display(Markdown(f"Datos listos: **{miles(len(ventas))} líneas de venta**, {len(clientes)} clientes, {len(productos)} productos. "
                 f"Fuente: {'archivos locales' if os.path.exists(BASE_LOCAL) else 'GitHub'}."))

## 1 · Seguridad de nivel de fila (RLS)
Cada jefe de sede tiene un rol con el filtro `Sedes[ciudad] = "…"`. Este cuaderno calcula lo que vería cada rol con **Modelado › Ver como**, para que compares con tu Power BI.

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Qué venta ve cada rol? |
> | **Qué se grafica** | Barras: venta neta 2025 visible para cada rol · Línea: lo que ve la gerencia (sin rol) |
> | **Por qué esta forma** | Comprueba de un vistazo que cada rol ve solo su parte y que nadie ve más que la gerencia. |
> | **Qué se descartó** | Revisar rol por rol sin comparar: un rol mal escrito (por ejemplo «Bogota» sin tilde) vería cero sin que nadie lo note. |
> | **Cómo se lee** | Las barras de los jefes suman exactamente la línea de la gerencia. Si un rol mostrara cero, el filtro está mal escrito. |
> | **En Excel 365** | Modelado › Administrar roles; luego Modelado › Ver como para cada rol. |

In [ ]:
# @title Gráfico 1 · Lo que ve cada rol
rol_mal_escrito = False  # @param {type:"boolean"}
v25 = validas[validas.anio == 2025]
roles = {f"Jefe {c}": (c if not (rol_mal_escrito and c == "Bogotá") else "Bogota") for c in ORDEN_CIUDADES}
visto = pd.Series({r: v25[v25.ciudad == f]["subtotal"].sum() for r, f in roles.items()}) / 1e6
total = v25.subtotal.sum() / 1e6
fig = go.Figure(go.Bar(x=visto.index, y=visto.values, marker_color=[MAGENTA if v == 0 else AZUL for v in visto.values],
                       text=[f"{v:,.0f}".replace(",", ".") for v in visto.values], textposition="outside"))
fig.add_hline(y=total, line_dash="dash", line_color=GRIS, annotation_text=f"Gerencia (sin rol): {millones(total * 1e6)}")
fig.update_layout(title=(f"Los seis roles suman {millones(visto.sum() * 1e6)}, igual a lo que ve la gerencia" if abs(visto.sum() - total) < 1
                         else f"Un rol ve $ 0: el filtro «Bogota» no coincide con «Bogotá»; faltan {millones((total - visto.sum()) * 1e6)}"),
                  yaxis_title=MM, height=380, yaxis_range=[0, total * 1.1])
fig.show()

## 2 · ¿Dónde se publica?

| Caso | Dónde | Por qué |
|---|---|---|
| Tablero con datos reales para la gerencia | **Servicio de Power BI** (área de trabajo) | Acceso por persona, actualización programada y RLS |
| Tu tablero con datos ficticios para la hoja de vida | **Tableau Public** | Es público: sirve como portafolio |
| Datos personales (salarios, clientes) | **No se publica** | Ley 1581 de 2012: datos personales protegidos |
| «Publicar en la web» con datos de la empresa | **No se publica** | Cualquiera con el enlace lo ve |

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Qué parte del informe ve cada jefe? |
> | **Qué se grafica** | Barras: porcentaje de las filas de Ventas que ve cada rol |
> | **Por qué esta forma** | Muestra que el rol no es un filtro visual: recorta las filas que el usuario puede consultar. |
> | **Qué se descartó** | Pensar que basta con ocultar un visual: quien exporta los datos los vería todos. |
> | **Cómo se lee** | Cada jefe ve entre el 7 % y el 30 % de las filas. Con RLS, exportar no le da acceso al resto. |
> | **En Excel 365** | Modelado › Ver como › rol; luego exporta los datos de un visual y revisa cuántas filas salen. |

In [ ]:
# @title Gráfico 2 · Filas visibles por rol
filas = v25.groupby("ciudad").size().reindex(ORDEN_CIUDADES) / len(v25)
fig = go.Figure(go.Bar(x=[f"Jefe {c}" for c in filas.index], y=filas.values, marker_color=TEAL, text=[porc(v) for v in filas.values], textposition="outside"))
fig.update_layout(title=f"Cada jefe ve solo su parte de las {miles(len(v25))} líneas de 2025", yaxis_tickformat=".0%", height=340, yaxis_range=[0, filas.max() * 1.25])
fig.show()

## 3 · Retos de interpretación

In [ ]:
# @title Reto 1 · ¿Qué venta neta 2025 ve el rol «Jefe Medellín»?
respuesta = "Elige una opción"  # @param ["Elige una opción", "2.906.529.105", "4.438.208.370", "14.692.379.210", "2.348.175.015"]
revisar(respuesta, miles(v25[v25.ciudad == "Medellín"].subtotal.sum()), """Es la barra de Medellín del gráfico 1; debe coincidir con «Ver como» en Power BI.""")

In [ ]:
# @title Reto 2 · ¿Dónde publicas tu tablero de portafolio con datos ficticios?
respuesta = "Elige una opción"  # @param ["Elige una opción", "Servicio de Power BI", "Tableau Public", "Por correo", "Publicar en la web con datos reales"]
revisar(respuesta, "Tableau Public", """Es público por diseño, ideal para el portafolio con datos ficticios.""")

In [ ]:
# @title Reto 3 · Un rol muestra $ 0. ¿Qué es lo más probable?
respuesta = "Elige una opción"  # @param ["Elige una opción", "No hay ventas", "El valor del filtro no coincide exactamente (tildes, espacios)", "Falta una medida", "El visual está mal"]
revisar(respuesta, "El valor del filtro no coincide exactamente (tildes, espacios)", """Activa «rol mal escrito» en el gráfico 1 para verlo.""")

## Apóyate en IA
Para diseñar los roles de seguridad:

```text
En Power BI, Sedes (ciudad) filtra a Ventas. Quiero que cada jefe vea solo su sede y la gerencia todo.
Propón los roles, una versión dinámica con USERPRINCIPALNAME y cómo probarlos con Ver como.
```

> **Verifica siempre.** Compara la respuesta de la IA con un dato que ya conozcas. No pegues datos personales reales (Ley 1581 de 2012): estos datos son ficticios, los de tu empresa no.

## Resumen
* En la empresa se publica en un área de trabajo, con actualización programada y roles.
* RLS recorta las filas; se prueba con Ver como y se verifica que los roles sumen el total.
* El portafolio va en Tableau Public, siempre con datos ficticios o públicos.

**Siguiente:** [Laboratorio · Taller P3](https://omarli-360.github.io/powerbi-tableau-analista/#taller-3)

<sub>Dr. Bits · ADNBITS S.A.S. · Contenido CC BY-SA 4.0 · Datos ficticios de Distribuidora Andina S.A.S.</sub>